# Item Burn Curve Smoothing Analysis

This notebook removes payment-row lumpiness by clustering short-gap postings, converting each `ITEMID` to a cumulative spend curve, and evaluating normalized expected-burn curve models.

## Method

1. Sort rows by `ITEMID, WPPostingDate`.
2. Collapse consecutive postings into the same cluster when the gap is `<= 3` days.
3. Sum `THIS_BURN` within each cluster.
4. Compute cumulative spend and cumulative spend percent per item.
5. Compute elapsed percent using the existing convention: `(elapsed_days_from_first_cluster + 30) / (last_cluster_date - first_cluster_date + 30)`.
6. Fit expected cumulative spend curves on train items and evaluate on held-out items.

This is retrospective because cumulative percent uses final item total. For production forecasting, the item total scale must be estimated from pre-completion metadata.

In [ ]:
import csv
from pathlib import Path

with open('ci_item_clustered_cumulative_curves.csv', newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(f'Cluster rows: {len(rows):,}')
print(reader.fieldnames)

## Data Reduction

| Metric | Value |
| --- | --- |
| Raw payment rows | 16,435 |
| Short-gap cluster threshold | 3 days |
| Clustered burn observations | 7,631 |
| Rows per cluster | 2.154 |
| Items | 875 |
| Train/test items | 679 / 196 |
| Median clusters per item | 7.00 |
| P90 clusters per item | 17.00 |
| Median modeled days per item | 239.04 |
| Median item total burn | 1,153,664.27 |

## Model Evaluation on Held-Out Items

Errors are in cumulative-spend-percent units. For example, MAE `0.10` means an average absolute error of 10 percentage points of final item spend.

| Model | MAE | RMSE | Median AE | P90 AE | Bias |
| --- | --- | --- | --- | --- | --- |
| Duration-bucket Beta CDF | 0.1492 | 0.2061 | 0.1179 | 0.3405 | 0.0009 |
| Pooled empirical median curve | 0.1514 | 0.2152 | 0.1028 | 0.3713 | 0.0039 |
| Cluster-count-bucket Beta CDF | 0.1518 | 0.2087 | 0.1112 | 0.3704 | 0.0028 |
| Global Beta CDF | 0.1554 | 0.2128 | 0.1191 | 0.3578 | 0.0032 |
| Linear cumulative spend | 0.1750 | 0.2492 | 0.1303 | 0.4286 | -0.1050 |

## Fitted Curve Parameters

| Parameter | Value |
| --- | --- |
| Global Beta CDF alpha | 0.489591 |
| Global Beta CDF beta | 0.759175 |
| Train MSE | 0.04242618 |

### Duration-Bucket Beta Parameters

| Duration bucket | alpha | beta |
| --- | --- | --- |
| 181-365d | 0.524573 | 0.822175 |
| 366-730d | 0.554490 | 1.004356 |
| <=180d | 0.804323 | 0.693961 |
| >730d | 0.587693 | 1.177911 |

### Cluster-Count-Bucket Beta Parameters

| Cluster-count bucket | alpha | beta |
| --- | --- | --- |
| 13+ clusters | 0.719111 | 1.244553 |
| 4-6 clusters | 0.253433 | 0.323358 |
| 7-12 clusters | 0.444078 | 0.699397 |
| <=3 clusters | 0.146929 | 0.152017 |

## Curve Plot

The points are sampled train clustered cumulative observations. The fitted global Beta curve and pooled empirical median curve are overlaid against a linear reference.



## Retrospective Per-Item Smoothers

These use each completed item’s full history, so they are smoothers rather than deployable cold-start forecasts. They are useful for estimating smoothed current burn/month after observing an item trajectory.

| Metric | Value |
| --- | --- |
| Per-item Beta CDF median MAE | 0.0349 |
| Per-item Beta CDF P90 MAE | 0.0683 |
| Per-item monotone PCHIP median interpolation MAE | 0.0000 |

## Rolling 30-Day Clustered Burn

A simple operational smoothed burn/month proxy is clustered spend in the trailing 30 days. This is noisy for sparse items but much less lumpy than raw postings.

| Quantile | Trailing 30-day clustered burn |
| --- | --- |
| 0.05 | 5,848.01 |
| 0.25 | 73,517.97 |
| 0.5 | 217,143.98 |
| 0.75 | 663,788.22 |
| 0.95 | 3,452,596.34 |

## Recommendation

For the business goal, use cumulative spend curves rather than row-level burn deltas. The best first production shape is a pooled expected cumulative curve plus item-specific updating:

```text
cluster short-gap postings <= 3 days
actual_position = cumulative_clustered_spend / estimated_item_total
expected_position = expected_curve(elapsed_pct, item_family/duration bucket)
position_delta = actual_position - expected_position
smoothed_burn_month = item_total_scale * derivative(expected_curve) / modeled_months
```

With only the current data, the strongest retrospective baseline is a pooled Beta/empirical cumulative curve. To make it a forward model, add item metadata and active-item snapshots so item total and expected duration can be estimated without using completed-item information.